# Lecture 4 Colab: Customer Segmentation with K-means

This notebook demonstrates unsupervised customer segmentation using synthetic customer behaviour data.


## Learning objectives

- Understand unsupervised learning.
- Generate synthetic customer behaviour data.
- Apply feature scaling before clustering.
- Run K-means clustering.
- Use elbow method and silhouette score.
- Interpret cluster profiles in a financial services context.


## Connection to Lecture 4

Supports unsupervised learning, K-means clustering, choosing the number of clusters, and helpful ML techniques.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
np.random.seed(42)


## Compatibility note for local Anaconda users

In Google Colab, `sklearn.cluster.KMeans` should normally run directly. Some local Windows/Anaconda environments may raise a `threadpoolctl` or OpenBLAS error when K-means starts. The helper below tries scikit-learn first and falls back to a small NumPy implementation only if that environment issue appears.


In [ ]:
def _numpy_kmeans(X, n_clusters, random_state=42, n_init=10, max_iter=300, tol=1e-4):
    """Small teaching fallback for environments where sklearn KMeans fails."""
    X = np.asarray(X, dtype=float)
    best_labels = None
    best_centers = None
    best_inertia = np.inf

    for init in range(n_init):
        rng = np.random.default_rng(random_state + init)
        center_idx = rng.choice(X.shape[0], size=n_clusters, replace=False)
        centers = X[center_idx].copy()

        for _ in range(max_iter):
            distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            labels = distances.argmin(axis=1)

            new_centers = centers.copy()
            for cluster_id in range(n_clusters):
                mask = labels == cluster_id
                if mask.any():
                    new_centers[cluster_id] = X[mask].mean(axis=0)
                else:
                    new_centers[cluster_id] = X[rng.integers(X.shape[0])]

            if np.max(np.abs(new_centers - centers)) < tol:
                centers = new_centers
                break
            centers = new_centers

        distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = distances.argmin(axis=1)
        inertia = float(np.sum(distances[np.arange(X.shape[0]), labels]))

        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels
            best_centers = centers

    return best_labels, best_inertia, best_centers


def fit_kmeans(X, n_clusters, random_state=42, n_init=10):
    """Use sklearn KMeans, with a NumPy fallback for local threadpool/OpenBLAS issues."""
    try:
        model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=n_init)
        labels = model.fit_predict(X)
        return labels, float(model.inertia_), model.cluster_centers_
    except AttributeError as exc:
        message = str(exc).lower()
        if "split" not in message and "threadpool" not in message and "openblas" not in message:
            raise
        print("Using NumPy fallback K-means due to a local sklearn/threadpoolctl issue.")
        return _numpy_kmeans(X, n_clusters=n_clusters, random_state=random_state, n_init=n_init)


## Dataset explanation

We generate four synthetic customer groups. The `true_group` column is included only to explain the synthetic data generation and is not used in clustering.


In [ ]:
def make_group(n, spend, tx_count, balance, digital, investment, label):
    return pd.DataFrame({
        "monthly_spend": np.clip(np.random.normal(spend, spend * 0.15, n), 0, None),
        "transaction_count": np.clip(np.random.normal(tx_count, tx_count * 0.18, n), 0, None),
        "average_balance": np.clip(np.random.normal(balance, balance * 0.20, n), 0, None),
        "digital_activity_score": np.clip(np.random.normal(digital, 8, n), 0, 100),
        "investment_activity_score": np.clip(np.random.normal(investment, 8, n), 0, 100),
        "true_group": label,
    })
parts = [
    make_group(200, 400, 12, 1800, 25, 15, "A_low_activity"),
    make_group(200, 900, 55, 4500, 82, 35, "B_digital_active"),
    make_group(200, 1400, 30, 18000, 58, 86, "C_high_value_investor"),
    make_group(200, 2600, 70, 3000, 65, 20, "D_high_spend_low_balance"),
]
df = pd.concat(parts, ignore_index=True)
feature_cols = ["monthly_spend", "transaction_count", "average_balance", "digital_activity_score", "investment_activity_score"]
df.head()


## Display and inspect data

A two-dimensional plot is useful for intuition, but clustering uses all selected features.


In [ ]:
display(df.head())
display(df[feature_cols].describe().round(2))
plt.figure(figsize=(7, 5))
plt.scatter(df["monthly_spend"], df["average_balance"], alpha=0.5)
plt.xlabel("Monthly spend")
plt.ylabel("Average balance")
plt.title("Synthetic customer data")
plt.grid(alpha=0.3)
plt.show()


## Why scaling matters

K-means uses distance. Large-scale features can dominate unless features are scaled.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])


## K-means with K = 4

K-means discovers groups without labels. It does not name the clusters for us.


In [ ]:
labels, inertia, centers = fit_kmeans(X_scaled, n_clusters=4, random_state=42, n_init=10)
df["cluster"] = labels
df[["true_group", "cluster"]].head()


## PCA visualization

PCA reduces the scaled features to two dimensions for visualization.


In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
plt.figure(figsize=(7, 5))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=df["cluster"], cmap="tab10", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters visualized with PCA")
plt.colorbar(scatter, label="Cluster")
plt.grid(alpha=0.3)
plt.show()


## Cluster profile table

Use average feature values to interpret cluster behaviour.


In [ ]:
cluster_profile = df.groupby("cluster")[feature_cols].mean().round(2)
cluster_profile


## Cluster interpretation

- Which cluster looks like digitally active customers?
- Which cluster may be high-value investment customers?
- Which cluster may need credit risk monitoring?
- Why are these names business interpretations rather than model outputs?


## Elbow method

The elbow is a heuristic, not a proof.


In [ ]:
inertias = []
for k in range(1, 11):
    labels, inertia, centers = fit_kmeans(X_scaled, n_clusters=k, random_state=42, n_init=10)
    inertias.append(inertia)
plt.figure(figsize=(7, 5))
plt.plot(range(1, 11), inertias, marker="o")
plt.xlabel("Number of clusters K")
plt.ylabel("Inertia")
plt.title("Elbow method")
plt.grid(alpha=0.3)
plt.show()


## Silhouette score

Silhouette measures compactness and separation. Higher is usually better, but business meaning still matters.


In [ ]:
rows = []
for k in range(2, 11):
    labels, inertia, centers = fit_kmeans(X_scaled, n_clusters=k, random_state=42, n_init=10)
    rows.append({"k": k, "silhouette": silhouette_score(X_scaled, labels)})
silhouette_df = pd.DataFrame(rows)
display(silhouette_df.round(3))
plt.figure(figsize=(7, 5))
plt.plot(silhouette_df["k"], silhouette_df["silhouette"], marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette score")
plt.title("Silhouette score by K")
plt.grid(alpha=0.3)
plt.show()


## Compare K values

Changing K can change cluster profiles and business interpretation.


In [ ]:
for k in [3, 5]:
    temp = df.copy()
    labels, inertia, centers = fit_kmeans(X_scaled, n_clusters=k, random_state=42, n_init=10)
    temp["cluster"] = labels
    print()
    print(f"Cluster profiles for K={k}")
    display(temp.groupby("cluster")[feature_cols].mean().round(2))


## What to observe

- Scaling changes distance calculations.
- K selection affects cluster profiles.
- Cluster names require human business interpretation.


## Common pitfalls

- Clusters are not automatically meaningful.
- K-means assumes roughly spherical clusters.
- Outliers can distort centroids.
- Feature scaling matters.
- Clustering is exploratory, not a final business policy by itself.


## Try it yourself

- Change K to 3 or 5.
- Remove feature scaling and compare results.
- Add a new feature such as `loan_balance`.
- Rename clusters based on average profiles.


## Reflection questions

- Why does K-means not need labels?
- Why can different K values lead to different business interpretations?
- What would you need before using clusters in a real bank campaign?


## Final summary

K-means groups customers by feature similarity. Scaling and K selection matter. Human interpretation is required.
